# ============================================================
# Author: Mayur Deshmukh
# Title: 01_raw_transform.ipynb
# Project: ML-Binary-Classifier-For-Stock-Price-Prediction
# Purpose: Raw Data Transformation
# Python Version: 3.11
# ============================================================

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
input_path = os.path.join('..', '..', 'input', 'all_stocks_5yr.csv')
df = pd.read_csv(input_path)
df.head()

In [ ]:
# ── Data Cleaning ──────────────────────────────────────────────────────────────

# 1. Parse date and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Name', 'date']).reset_index(drop=True)

# 2. Drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")

# 3. Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Drop rows where price or volume data is missing
df = df.dropna(subset=['open', 'high', 'low', 'close', 'volume'])

# 4. Remove rows with non-positive prices or volume
invalid_mask = (df[['open', 'high', 'low', 'close']] <= 0).any(axis=1) | (df['volume'] < 0)
print(f"\nInvalid price/volume rows removed: {invalid_mask.sum()}")
df = df[~invalid_mask].reset_index(drop=True)

# 5. Sanity check: high >= low
inconsistent = df[df['high'] < df['low']]
print(f"Rows where high < low: {len(inconsistent)}")
df = df[df['high'] >= df['low']].reset_index(drop=True)

# 6. Correct dtypes
df['volume'] = df['volume'].astype('int64')
df['Name'] = df['Name'].astype('category')

print(f"\nCleaned dataset shape: {df.shape}")
df.dtypes

In [ ]:
# ── Target Label: Price Direction ──────────────────────────────────────────────

# Compare each day's close to the previous day's close, grouped by ticker
df['target'] = (df.groupby('Name')['close']
                  .diff()
                  .gt(0)
                  .astype(int))

# First row per ticker has no previous day — drop it
df = df.dropna(subset=['target']).reset_index(drop=True)

print(f"Label distribution:\n{df['target'].value_counts()}")
print(f"\nLabel balance: {df['target'].mean():.2%} up days")
df[['date', 'Name', 'close', 'target']].head(10)

In [ ]:
# ── Label Distribution per Ticker ──────────────────────────────────────────────

label_dist = (df.groupby('Name')['target']
                .value_counts(normalize=True)
                .mul(100)
                .round(2)
                .rename('percentage')
                .reset_index())

label_dist['target'] = label_dist['target'].map({1: 'Up', 0: 'Down'})
label_dist = label_dist.pivot(index='Name', columns='target', values='percentage').reset_index()
label_dist.columns.name = None

print(f"Label distribution (%) per ticker — showing top 5:")
display(label_dist.head())

print(f"\nFull dataset (top 5 rows):")
display(df.head())

In [ ]:
# ── Filter: Top N Stocks by Record Count (max 25K total) ───────────────────────

record_counts = df.groupby('Name').size().sort_values(ascending=False)

top10 = record_counts.head(10)
top9  = record_counts.head(9)

if top10.sum() <= 25000:
    selected_tickers = top10.index.tolist()
    print(f"Top 10 stocks selected (total records: {top10.sum()})")
else:
    selected_tickers = top9.index.tolist()
    print(f"Top 10 exceeded 25K records ({top10.sum()}), falling back to top 9 (total records: {top9.sum()})")

print(f"\nSelected tickers: {selected_tickers}")
print(f"\nRecord counts:")
print(record_counts[selected_tickers])

df_filtered = df[df['Name'].isin(selected_tickers)].reset_index(drop=True)

print(f"\nFiltered dataset shape: {df_filtered.shape}")
df_filtered.head()

In [ ]:
# ── Export Filtered Dataset ─────────────────────────────────────────────────────

output_path = os.path.join('..', '..', 'output', 'filtered_stocks.csv')
df_filtered.to_csv(output_path, index=False)
print(f"Exported {df_filtered.shape[0]} rows to: {output_path}")